# Practical 6B — Pascal VOC YOLO and SSD MobileNet

## Aim
Demonstrate YOLO object detection and SSD MobileNet inference on Pascal VOC images.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Requirements\nRequires Ultralytics, PyTorch, torchvision and internet access for weights and VOC data. The pretrained models are COCO-trained; this notebook is a workflow demonstration, not a Pascal VOC accuracy benchmark.

## 1. Install object-detection dependencies\nYOLOv8 is used as the runnable YOLO implementation. SSD MobileNet is demonstrated using torchvision's pretrained SSD model. Internet is needed for dataset/model weights.

In [ ]:
%pip install -q ultralytics torch torchvision pillow matplotlib

## 2. Download Pascal VOC 2007 validation split

In [ ]:
from torchvision.datasets import VOCDetection
dataset=VOCDetection(root="./data",year="2007",image_set="val",download=True)
print("Images:",len(dataset))

## 3. YOLOv8 inference on a VOC image\nUltralytics pretrained weights are COCO-trained, not VOC-trained. This illustrates the detector workflow; class overlap is incomplete.

In [ ]:
from ultralytics import YOLO
import tempfile, os, matplotlib.pyplot as plt
model=YOLO("yolov8n.pt")
image,annotation=dataset[0]
tmp="voc_sample.jpg"; image.save(tmp)
results=model.predict(source=tmp,conf=0.25,verbose=False)
annotated=results[0].plot()[:,:,::-1]
plt.figure(figsize=(10,8)); plt.imshow(annotated); plt.axis("off"); plt.show()
print(results[0].names)

## 4. SSD MobileNet inference

In [ ]:
import torch, torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.transforms.functional import to_tensor, to_pil_image
from torchvision.utils import draw_bounding_boxes
weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
ssd=ssdlite320_mobilenet_v3_large(weights=weights).eval()
device="mps" if torch.backends.mps.is_available() else "cpu"; ssd.to(device)
tensor=to_tensor(image).to(device)
with torch.no_grad(): pred=ssd([tensor])[0]
keep=pred["scores"].cpu()>0.5
boxes=pred["boxes"].cpu()[keep]; labels=pred["labels"].cpu()[keep]; scores=pred["scores"].cpu()[keep]
cats=weights.meta["categories"]
texts=[f"{cats[int(l)]} {float(s):.2f}" for l,s in zip(labels,scores)]
draw=draw_bounding_boxes((tensor.cpu()*255).byte(),boxes,labels=texts,width=2)
plt.figure(figsize=(10,8)); plt.imshow(to_pil_image(draw)); plt.axis("off"); plt.show()

## 5. Compare YOLO and SSD MobileNet

| Detector | Characteristics |
|---|---|
| YOLO | Single-stage detector; predicts boxes and classes in one network pass, designed for speed. |
| SSD MobileNet | Single-shot multibox detection with a MobileNet backbone, designed for efficient inference. |

Both pretrained models above are COCO-trained. Pascal VOC labels/classes do not exactly match COCO, so predictions are demonstrations and not a valid VOC benchmark. For a fair comparison, train/fine-tune both on the same VOC split and evaluate with mAP at a specified IoU threshold.

## 6. Conclusion\nCompare detection speed qualitatively, bounding-box quality and class predictions. For quantitative evaluation, use a VOC-compatible mAP evaluator and models trained on the same dataset.